In [ ]:
from dlfs.layers import DenseLayer
from dlfs.activation import ReLU
from dlfs.loss import MSE_Loss
from dlfs.optimizers import Optimizer_SGD, Optimizer_Adam
from dlfs.model import SequentialModel

from viz_helpers import *

# Dense Layer visualisation for regression

- this notebook aims to visualize 1D, 2D, 3D linear transformations (by Dense Layers) and activation functions applied to regression data 

# Different datasets for visualisation

In [ ]:
from sklearn.datasets import make_regression

def make_sine():
    X = np.linspace(-5, 5, 100)
    y = 2*np.sin(X) + np.random.randn(100) * 0.3
    return X, y

def make_cubic():
    X = np.linspace(-20, 20, 50)
    noise = np.random.normal(0, 0.5, size=X.shape)
    y = X.flatten()**3 - 2 * X.flatten()**2 + 3 * X.flatten() + 1 + noise.flatten()
    return X, y

def make_logarithm():
    X = np.random.rand(100, 1) * 10
    noise = np.random.normal(0, 0.1, size=X.shape)
    y = np.log(X.flatten() + 1) + noise.flatten()
    return X, y

dataset_dict = {
    "simple": make_regression(n_samples=200, n_features=1, noise=20),
    "sine": make_sine(),
    "cubic": make_cubic(),
    "logarithm": make_logarithm()
    }

# Simple regression dataset

In [ ]:
from sklearn.preprocessing import StandardScaler

dataset = "logarithm"
X, y = dataset_dict[dataset]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

ax[0].scatter(X, y, c=y, edgecolors="k", alpha=0.7)
ax[0].set_title("Original data")

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X.reshape(-1, 1))
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

ax[1].scatter(X_scaled, y_scaled, c=y_scaled, edgecolors="k", alpha=0.7)
ax[1].set_title("Normalized data")

plt.show()

# Training regression model

In [ ]:
layers = [DenseLayer(1, 3), 
          ReLU(),
          DenseLayer(3, 2),
          ReLU(),
          DenseLayer(2, 1)]

model = SequentialModel(layers=layers, loss_function=MSE_Loss(), optimizer=Optimizer_SGD(learning_rate=1e-3, momentum=0.3, decay=0.))

model.train(X_scaled, y_scaled, print_every=500, epochs=1000)

# Extract each layer's output

In [ ]:
model.forward(X_scaled)

Z1 = model.wrapper.layers[0].output.copy() # first dense layer (N, 2)
A1 = model.wrapper.layers[1].output.copy() # first dense + relu (N, 2)
Z2 = model.wrapper.layers[2].output.copy() # second dense (N, 1)
A2 = model.wrapper.layers[3].output.copy() # first dense layer (N, 2)
Z3 = model.wrapper.layers[4].output.copy() # first dense + relu (N, 2)

# Plotting original 1D data on a number line

In [ ]:
plot_1d_regression_output(X, y, "Original 1D input data")

# Plotting first linear transformation

In [ ]:
plot_3d_regression_output(Z1, y_scaled.reshape(-1), "1D -> 3D using first Dense Layer")

# Plotting first linear transformation and first activation

In [ ]:
plot_3d_regression_output(A1, y_scaled.reshape(-1), "1D -> 3D using first Dense Layer + ReLU")

# Plotting second linear transformation

In [ ]:
plot_2d_regression_output(Z2, y_scaled.reshape(-1), "3D -> 2D using second Dense Layer")

# Plotting second linear transformation and second activation

In [ ]:
plot_2d_regression_output(A2, y_scaled.reshape(-1), "3D -> 2D using second Dense Layer + ReLU")

# Plotting third linear transformation

In [ ]:
plot_1d_regression_output(Z3, y_scaled.reshape(-1), "2D -> 1D using third Dense Layer")

# Preprocess input data for final plot

In [ ]:
X_range = np.linspace(X.min(), X.max(), 50)
X_range_scaled = scaler_X.transform(X_range.reshape(-1, 1))
y_pred_scaled = model.predict(X_range_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

sorted_X = np.sort(X)
sorted_scaled_X = scaler_X.transform(sorted_X.reshape(-1, 1))
y_pred_sorted_scaled = model.predict(sorted_scaled_X)
y_pred_sorted = scaler_y.inverse_transform(y_pred_sorted_scaled)

# Visualize network output

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,6))
ax = ax.flatten()

ax[0].scatter(X, y, c=y)
ax[0].set_title("Original X and y")
ax[0].plot(X_range, y_pred, color="red", label="NN prediction", linewidth=2)

ax[1].scatter(sorted_X, y_pred_sorted, c=y_scaled)
ax[1].plot(X_range, y_pred, color="red", label="NN prediction", linewidth=2)
ax[1].set_title("Original X and predicted y")

ax[0].legend()
ax[1].legend()

plt.show()